# Authority deference v0.1

Seven-arm experiment on conflicting UK legal information: behavioural measurement, residual-stream divergence, and bidirectional activation patching, plus behavioural-only API arms when keys are present.

Requires a GPU with ≥40 GB (Colab A100 40GB). A T4 is not sufficient. Default local checkpoint `google/gemma-3-12b-it` is gated: accept the Hugging Face licence and set `HF_TOKEN`. Ungated alternative: `Qwen/Qwen3-4B` in the config cell.

Interpretability runtime: [interp-engine](https://www.neuronpedia.org/blog/interp-engine). Protocol: `RESEARCH.md`. This notebook only sets up the runtime and calls `src/`.


## 1. Clone


In [ ]:
import os, sys, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes — stashing:\n" + dirty)
        !git -C $REPO_DIR stash -u
        print("Recover with: !git -C $REPO_DIR stash pop")
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

print("working directory:", pathlib.Path.cwd())
!git -C $REPO_DIR log --oneline -1


## 2. Install

Eager backend. vLLM requires `enforce_eager=True` to capture activations, which removes most of its speed advantage at this size (`RESEARCH.md` §3.1).

If Colab asks to restart after pip (numpy/torch version change), restart and continue from this cell. `src/` is already on `sys.path`.


In [ ]:
# '.[apis]' adds the OpenAI and Anthropic SDKs. Colab ships openai but not anthropic.
%pip install -q -e '.[apis]'
print("installed")

## 3. GPU


In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU, then Run All."
props = torch.cuda.get_device_properties(0)
print(f"{props.name}  |  {props.total_memory / 1e9:.1f} GB  |  compute {props.major}.{props.minor}")

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("dtype:", DTYPE, " device:", DEVICE)


## 4. interp-engine

Engine and `transformers` versions are recorded in `manifest.json`; the engine hooks `transformers` modules and has no independent forward pass.


In [ ]:
import interp_engine, transformers
from interp_engine import CAPABILITIES

print("interp-engine", interp_engine.__version__)
print("transformers ", transformers.__version__)
print("vLLM backend available:", interp_engine.vllm_installed())

## 5. Config


In [ ]:
from microscope.experiment import RunConfig, run_all
from microscope.scenarios import ARMS, DEFAULT_CONTRAST

# "sweep"  — section 7b, all configured models
# "single" — section 7, `cfg` only
# "reuse"  — newest completed run on disk
MODE = "sweep"

OPENAI_MODEL = "gpt-5.1"
ANTHROPIC_MODEL = "claude-opus-5"

MODEL_ID = "Qwen/Qwen3-14B"   # ungated alternative: "Qwen/Qwen3-4B"

cfg = RunConfig(
    model_id=MODEL_ID,
    provider="local",
    backend="eager",
    dtype=DTYPE,
    extra_load_kwargs={"device": DEVICE},
    n_candidate_layers=4,
    # arms=("floor", "junior_said", "partner_said"),
    # contrast=DEFAULT_CONTRAST,
    # limit=5,
)

for arm in ARMS:
    cue = arm.cue or "(no assertion)"
    print(f"{arm.name:20s} {cue:40s} {arm.note}")
print()
print("mechanistic contrast:", cfg.contrast)


### Hugging Face token (gated checkpoints)


In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as exc:
    print(f"No HF_TOKEN ({exc}). Required for gated checkpoints.")


## 6. Scenarios

30 England-and-Wales items. Arms differ only in the attribution of the false proposition.


In [ ]:
from microscope.scenarios import load_scenarios, ARMS
import pandas as pd

scenarios = load_scenarios()
print(len(scenarios), "scenarios")
display(pd.Series([s.area for s in scenarios]).value_counts().rename("scenarios").to_frame())

example = scenarios[0]
print(example.prompt("floor"))

print("\n" + "=" * 78)
print("ADDITIONAL INFORMATION, by arm:\n")
for arm in ARMS:
    if arm.cue:
        print(f"  {arm.name:20s} {arm.cue}")
        print(f"  {'':20s} {example.false_proposition}\n")


## 7. Single-model run

Behaviour, residual capture, then bidirectional patching with controls. ~20–40 min on A100 for 30 scenarios.

`run_all` is submitted on a worker thread: interp-engine's sync facade cannot run inside Colab's kernel event loop.


In [ ]:
import concurrent.futures
from pathlib import Path

def _newest_completed():
    done = sorted(p for p in Path("results").glob("*Z") if (p / "summary.json").exists())
    if not done:
        raise SystemExit("No completed run on disk. Set MODE to 'single' or 'sweep'.")
    return done[-1]

if MODE == "single":
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        run_dir = pool.submit(run_all, cfg).result()
elif MODE == "reuse":
    run_dir = _newest_completed()
    print(f"Reusing {run_dir}")
else:
    run_dir = None
    print("MODE is 'sweep' — skipping; section 7b runs the sweep.")

run_dir


### API keys

Must be loaded before the sweep. The sweep includes a provider only if its key is already in the environment.


In [ ]:
for secret in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        value = userdata.get(secret)
    except Exception as exc:
        print(f"{secret}: unavailable ({type(exc).__name__}) — skipped")
        continue
    if value:
        os.environ[secret] = value
        print(f"{secret}: loaded ({len(value)} chars)")
    else:
        print(f"{secret}: empty — skipped")


## 7b. Sweep

One session, models sequential, GPU released between loads. Same git revision for the whole comparison.

| config | role |
| --- | --- |
| Qwen3-14B, thinking on | generate-mode parse; run first so a parse failure surfaces before the mechanistic jobs |
| Qwen3-14B, thinking off | paired logits-mode / mechanistic condition |
| gemma-3-12b-it | second open family; no reasoning template control |
| gpt-5.1 / claude-opus-5 | behavioural-only API arms, if keys are set |

Reasoning-on local runs skip experiments 2–4: the answer is no longer at the final prompt position.


In [ ]:
import json
from microscope.experiment import run_sweep, compare_runs

def local(model_id, thinking=False):
    return RunConfig(model_id=model_id, provider="local", backend="eager", dtype=DTYPE,
                     extra_load_kwargs={"device": DEVICE}, n_candidate_layers=4,
                     enable_thinking=thinking)

runs = {}
if MODE != "sweep":
    print(f"MODE is {MODE!r} — skipping the sweep.")
else:
    sweep = [
        local("Qwen/Qwen3-14B", thinking=True),
        local("Qwen/Qwen3-14B", thinking=False),
        local("google/gemma-3-12b-it"),
    ]
    for model_id, provider, opts in [(OPENAI_MODEL, "openai", {}),
                                     (ANTHROPIC_MODEL, "anthropic", {"effort": "low"})]:
        if os.environ.get(f"{provider.upper()}_API_KEY"):
            sweep.append(RunConfig(model_id=model_id, provider=provider, provider_options=opts))
        else:
            print(f"SKIPPING {model_id}: no {provider.upper()}_API_KEY")

    print("Sweep:")
    for c in sweep:
        print(f"  - {c.model_id} ({c.provider}"
              + (", thinking" if c.enable_thinking else "") + ")")
    print()

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        runs = pool.submit(run_sweep, sweep).result()

    for path in runs.values():
        if json.loads((path / "manifest.json").read_text()).get("mechanistic"):
            run_dir = path
            break
    else:
        run_dir = next(iter(runs.values()), None)
    print(f"Detail sections will use: {run_dir}")

runs


In [ ]:
table = compare_runs(runs)
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

if "Qwen/Qwen3-14B (thinking)" in table.index and "Qwen/Qwen3-14B (no thinking)" in table.index:
    on = table.loc["Qwen/Qwen3-14B (thinking)", "partner_confirmed"]
    off = table.loc["Qwen/Qwen3-14B (no thinking)", "partner_confirmed"]
    print(f"Qwen3-14B partner_confirmed: reasoning off {off:.0%} → on {on:.0%}")


## 8. Results


In [ ]:
import json

summary = json.loads((run_dir / "summary.json").read_text())
print(json.dumps(summary["behavioural"], indent=2))

### Quality report

Written by `run_all` to `quality_report.json`. Checks this run's outputs (parse rate, floor accuracy, zero-patch no-op, etc.), not the pipeline. `FAIL`: do not treat the figures as findings. Definitions: `src/microscope/quality.py`.


In [ ]:
from microscope import quality

report = json.loads((run_dir / "quality_report.json").read_text())
print(quality.format_report(report))
if report["overall"] == "fail":
    print("\nFAIL — see checks above.")


In [ ]:
print(json.dumps(summary["intervention_controls"], indent=2))


In [ ]:
from IPython.display import Image, display

for figure in sorted((run_dir / "plots").glob("*.png")):
    print(figure.name)
    display(Image(str(figure)))

## API arms (if not already in the sweep)

Closed-weight backends are behavioural-only (`LocalBackend` is required for capture and patching).

Anthropic exposes no token logprobs, so the cross-model endpoint is FPAR (chosen letter). `samples=k` estimates a proportion at k× cost. Reasoning effort is a separate experimental factor (`reasoning_effort` / `effort`).


In [ ]:
# One prompt per provider, before committing to ~210 calls. A wrong model id or an auth
# problem surfaces here in seconds rather than halfway through a run.
from microscope.backends import BackendSpec
from microscope.scenarios import load_scenarios


if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        names = sorted(m.id for m in OpenAI().models.list())
        print(f"OpenAI models available ({len(names)}), a sample:")
        print("  " + ", ".join(n for n in names if n.startswith(("gpt", "o")))[:400])
        print(f"\n  '{OPENAI_MODEL}' available: {OPENAI_MODEL in names}")
    except Exception as exc:
        print(f"OpenAI model listing failed: {type(exc).__name__}: {exc}")

probe = load_scenarios()[0].prompt("partner_said")
for kind, model_id, opts in [
    ("openai", OPENAI_MODEL, {}),
    ("anthropic", ANTHROPIC_MODEL, {"effort": "low"}),
]:
    if not os.environ.get(f"{kind.upper()}_API_KEY"):
        print(f"\n{kind}: no key, skipping")
        continue
    try:
        m = BackendSpec(kind=kind, model_id=model_id, options=opts).build().measure(probe)
        print(f"\n{kind} / {model_id}")
        print(f"  letter={m.chosen_letter!r}  parsed={m.parse_ok}  probs={m.probability_source}")
        print(f"  said: {m.generated[:120]!r}")
    except Exception as exc:
        print(f"\n{kind} / {model_id} FAILED: {type(exc).__name__}: {exc}")

In [ ]:
api_runs = {}

if MODE == "sweep":
    # The sweep already ran these; running them again would just spend the API budget twice.
    print("MODE is 'sweep' -- API arms already covered above.")
else:

    api_configs = [
        RunConfig(model_id=OPENAI_MODEL, provider="openai"),
        RunConfig(model_id=ANTHROPIC_MODEL, provider="anthropic",
                  provider_options={"effort": "low"}),
        # Reasoning as a variable -- does working through the statute catch the conflict?
        # RunConfig(model_id=ANTHROPIC_MODEL, provider="anthropic",
        #           provider_options={"effort": "high"}),
    ]

    for api_cfg in api_configs:
        if not os.environ.get(f"{api_cfg.provider.upper()}_API_KEY"):
            print(f"skipping {api_cfg.model_id}: no key")
            continue
        label = f"{api_cfg.model_id} ({api_cfg.provider_options.get('effort', 'default')})"
        try:
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
                api_runs[label] = pool.submit(run_all, api_cfg).result()
        except Exception as exc:
            print(f"{label} failed: {type(exc).__name__}: {exc}")

    api_runs

In [ ]:
# FPAR per arm is comparable across every provider -- it needs only the answer letter,
# which is why it is the primary cross-model measure. See RESEARCH.md.
rows = []
for label, path in [(json.loads((run_dir / "manifest.json").read_text())["model"], run_dir),
                    *api_runs.items()]:
    s = json.loads((path / "summary.json").read_text())
    rows.append({"model": label, **s["behavioural"]["fpar_by_arm"]})

table = pd.DataFrame(rows).set_index("model")
display(table.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

## 9. Outputs

`summary.json` and `quality_report.json`. Behavioural / representational / causal claims are distinct; see `RESEARCH.md`.

## 10. Export

Colab deletes the runtime disk on disconnect. `manifest.json` records checkpoint revision, engine version, `transformers` version, git, GPU, and seed.


In [ ]:
import shutil

archive = shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir)
print(archive)

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"Download it from the file browser instead ({exc}).")